In [1]:
import os

In [2]:
os.getcwd()

'/home/thienhb/Workspace/med-doc-retrieval/notebooks'

In [3]:
os.chdir("../")

In [4]:
os.getcwd()

'/home/thienhb/Workspace/med-doc-retrieval'

In [5]:
os.listdir('.')

['.env',
 '.python-version',
 'outputs',
 '.pytest_cache',
 'notebooks',
 '.gitignore',
 'pyproject.toml',
 '.ruff_cache',
 'README.md',
 '.git',
 'src',
 'scripts',
 'data',
 '.env.example',
 'configs',
 '.venv',
 'tests',
 'main.py',
 'competition_guide.md',
 'uv.lock',
 'TEAM_GUIDE.md']

In [109]:
import pandas as pd

# Read a single Parquet file into a DataFrame
df = pd.read_parquet("data/processed/parquet_corpus/corpus_part_000.parquet")
df1 = pd.read_parquet("data/processed/parquet_corpus/corpus_part_001.parquet")
df2 = pd.read_parquet("data/processed/parquet_corpus/corpus_part_002.parquet")
df3 = pd.read_parquet("data/processed/parquet_corpus/corpus_part_003.parquet")
df4 = pd.read_parquet("data/processed/parquet_corpus/corpus_part_004.parquet")
df5 = pd.read_parquet("data/processed/parquet_corpus/corpus_part_005.parquet")
corpus = pd.read_parquet("data/raw/vibio_mir/links_corpus.parquet")

# Show the first few rows
print(df.head())

    doc_id                                                url  \
0   733125  https://www.vinmec.com/vie/bai-viet/-u-malt-da...   
1  2989609  https://www.a-hospital.com/w/%CE%91%E5%9C%B0%E...   
2   323036  https://suckhoecongdongonline.vn/0h-hom-nay-26...   
3  2474097           https://baidianfeng.familydoctor.com.cn/   
4  3984397      https://www.youlai.cn/yyk/article/136046.html   

                                               title  \
0  U malt dạ dày Phần 1: Tổng quan, chẩn đoán và ...   
1                                              地中海贫血   
2  0h hôm nay 26/7 công bố điểm thi THPT Quốc gia...   
3                                         鐨偆绉戦閬撀风櫧鐧滈   
4                                   免疫性血小板减少性紫癜-有来医生   

                                                text lang  \
0  Bài viết bởi Bác sĩ Mai Viễn Phương - Khoa Khá...   vi   
1  地中海贫血\n| A+医学百科 >> 地中海贫血 | \n地中海贫血又称海洋性贫血。是一组遗...   zh   
2  Tra cứu điểm thi THPT Quốc gia 2021\nSau 0h hô...   vi   
3  鐧界櫆椋庝笉鑳藉悆鍚淮c鐨勯鐗╋紵\n鐧界櫆椋庢槸

In [110]:
corpus.head()

,id,url
0,583,http://special.vietnamplus.vn/benh_vien_nhan_ai
1,584,https://bachmai.gov.vn/bai-viet/-nguoi-dan-se-...
2,585,https://bachmai.gov.vn/bai-viet/-roi-loan-su-t...
3,586,https://bachmai.gov.vn/bai-viet/1-4-nguoi-truo...
4,587,https://bachmai.gov.vn/bai-viet/1-5-phai-manh-...


In [7]:
df.iloc[0]

doc_id                                                   733125
url           https://www.vinmec.com/vie/bai-viet/-u-malt-da...
title         U malt dạ dày Phần 1: Tổng quan, chẩn đoán và ...
text          Bài viết bởi Bác sĩ Mai Viễn Phương - Khoa Khá...
lang                                                         vi
domain                                           www.vinmec.com
char_count                                                 7110
status                                                  success
Name: 0, dtype: object

In [8]:
import re
import unicodedata

def clean_medical_text(raw_text: str) -> str:
    # 1. Chuẩn hóa Unicode tiếng Việt về dạng NFC
    text = unicodedata.normalize('NFC', raw_text)
    
    # 2. Xóa phần giới thiệu tác giả/bệnh viện ở đầu bài
    # Ví dụ: "Bài viết bởi Bác sĩ... - Bệnh viện..."
    text = re.sub(r"^Bài viết bởi.*?(?=\n\n|\n[A-Z0-9])", "", text, flags=re.IGNORECASE | re.DOTALL)
    
    # 3. Xóa các câu điều hướng, lời mời xem tiếp ở cuối
    # Ví dụ: "Mời quý khách xem tiếp:...", "Mời quý vị xem tiếp:..."
    text = re.sub(r"Mời quý (khách|vị) xem tiếp:.*", "", text, flags=re.IGNORECASE)
    
    # 4. Xóa phần danh sách tài liệu tham khảo (References)
    # Thường bắt đầu bằng "Nguồn tham khảo :" hoặc các tên tác giả định dạng dạng "- Tên tác giả. Năm."
    text = re.sub(r"Nguồn tham khảo\s*:.*", "", text, flags=re.IGNORECASE | re.DOTALL)
    
    # 5. Xóa các dòng rác hoặc kêu gọi đặt lịch ở chân trang (nếu có)
    text = re.sub(r"Để đặt lịch khám tại viện.*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"Tải và đặt lịch khám tự động.*", "", text, flags=re.IGNORECASE)

    # 6. Chuẩn hóa khoảng trắng và ngắt dòng thừa
    # Thay thế các khoảng trắng liên tiếp bằng 1 khoảng trắng
    text = re.sub(r"[ \t]+", " ", text)
    # Loại bỏ các dòng trống thừa (nhiều hơn 2 lần xuống dòng liên tiếp)
    text = re.sub(r"\n\s*\n", "\n\n", text)
    
    return text.strip()


In [17]:
df.iloc[3]["url"]

'https://baidianfeng.familydoctor.com.cn/'

In [10]:
df.iloc[0]["title"]

'U malt dạ dày Phần 1: Tổng quan, chẩn đoán và triệu chứng'

In [11]:
df.iloc[0]["text"]

'Bài viết bởi Bác sĩ Mai Viễn Phương - Khoa Khám bệnh & Nội khoa - Bệnh viện Đa khoa Quốc tế Vinmec Central Park\nBệnh lý u lympho xảy ra ở nhiều cơ quan trong cơ thể, nhưng có một bệnh lý u lympho liên quan đến niêm mạc dạ dày do Helicobacter Pylori, mà ít được lưu tâm, vấn đề đó sẽ được trình bày trong bài này.\n1. Định vị dạ dày trong cơ thể người\nDạ dày là phần đầu tiên của hệ tiêu hóa nằm bên trong ổ bụng.\nDạ dày hay bao tử là nơi phình to nhất của hệ thống đường tiêu hóa trong cơ thể con người, có cấu tạo phức tạp, thực hiện hai chức năng chính là nghiền cơ học thức ăn, thấm dịch vị và phân hủy thức ăn nhờ hệ enzyme tiêu hóa trong dịch vị. Mặc dù là một phần của cơ thể thế nhưng không phải ai cũng biết dạ dày là gì? Dạ dày là một phần quan trọng trong hệ tiêu hóa của con người, là bộ phận nằm giữa thực quản và tá tràng, phần đầu của ruột non. Đây được xem là bộ phận tiêu hóa lớn nhất có chức năng chứa và tiêu hóa thực phẩm.\n2. Hạch bạch huyết (hạch lympho) là gì?\nHạch bạch hu

In [12]:
clean_medical_text(df.iloc[0]["text"])

'Bệnh lý u lympho xảy ra ở nhiều cơ quan trong cơ thể, nhưng có một bệnh lý u lympho liên quan đến niêm mạc dạ dày do Helicobacter Pylori, mà ít được lưu tâm, vấn đề đó sẽ được trình bày trong bài này.\n1. Định vị dạ dày trong cơ thể người\nDạ dày là phần đầu tiên của hệ tiêu hóa nằm bên trong ổ bụng.\nDạ dày hay bao tử là nơi phình to nhất của hệ thống đường tiêu hóa trong cơ thể con người, có cấu tạo phức tạp, thực hiện hai chức năng chính là nghiền cơ học thức ăn, thấm dịch vị và phân hủy thức ăn nhờ hệ enzyme tiêu hóa trong dịch vị. Mặc dù là một phần của cơ thể thế nhưng không phải ai cũng biết dạ dày là gì? Dạ dày là một phần quan trọng trong hệ tiêu hóa của con người, là bộ phận nằm giữa thực quản và tá tràng, phần đầu của ruột non. Đây được xem là bộ phận tiêu hóa lớn nhất có chức năng chứa và tiêu hóa thực phẩm.\n2. Hạch bạch huyết (hạch lympho) là gì?\nHạch bạch huyết (hạch lympho) là một phần của hệ bạch huyết, nằm trong vô số các cấu trúc trơn, hình bầu dục dẹp, rải rác dọc

In [ ]:
vi_docs = 

In [24]:
for line in df.iloc[10]["text"].split("\n"):
    print(line)

关键词： 平滑肌组织瘤；平滑肌肉瘤；腔外型；胃
【摘要】目的总结腔外型胃平滑肌肿瘤的诊断及治疗经验。方法回顾性分析过去20年48例腔外型胃平滑肌肿瘤的临床特点、辅助检查的准确率以及手术方式。结果48例腔外型胃平滑肌肿瘤占同期胃平滑肌肿瘤的68.6％，主要临床表现为腹部包块37.9％，消化道出血37.9％及腹部不适33.3％。术前胃肠钡餐及胃镜检查准确率分别为57.9％及41.7％，术前确诊率为49.4％。手术以胃楔形切除及胃大部分切除为主。结论腔外型胃平滑肌肿瘤早期缺乏特异性临床表现，易误诊。胃肠钡餐及胃镜联合B超和CT可提高确诊率。治疗上以胃楔形切除及胃大部分切除为主，不必常规清扫淋巴结。
Exogastric tumors of gastric smooth muscle
HE Yulong, LIU Tongcheng, ZHENG Zhangqing, et al
(Division of Gastrointestinal and Pancreatic Surgery, First Affiliated Hospital, Sun Yat sen University of Medical Scienses,Guangzhou 510080,China)
【Abstract】 Objective To investigate the clinical features, diagnostic methods and treatment principles of exogastric gastric smooth muscle tumor. Method Clinical presentations, diagnostic procedures and surgical treatments of 48 cases with exogastric gastric smooth muscle tumor during past 20 years were analyzed retrospectively.Results Fourty eight cases of exogastric gastric smooth muscle tumors accounted for 68. 6％ of all the patients with gastric smooth

In [13]:
len(df.iloc[0]["text"])

7110

In [97]:
from src.ingestion.chunker import DocumentChunker                                                                                                                                                                                                            

In [ ]:
# 2. Khởi tạo Chunker (tự động ngắt theo câu/đoạn tự nhiên, không ngắt cụt từ)                                                                                                                               
chunker = DocumentChunker(                                                                                                                                                                                   
    max_chunk_size=1200,    # Độ dài tối đa mỗi chunk (ký tự)                                                                                                                                                 
    chunk_overlap=200,      # Độ gối đầu giữa 2 chunk liên tiếp (để không mất ngữ cảnh)                                                                                                                       
    split_by_sentences=True # Ưu tiên ngắt tại dấu chấm, hỏi, than
)

In [96]:
# 3. Thực hiện tách chunk                                                                                                                                                                                    
chunks = chunker.chunk_document(doc_id="doc_123", text=df.iloc[31]["text"], lang="vi")
# 4. Xem kết quả
print(f"Tổng số chunks tạo ra: {len(chunks)}\n")
for i, c in enumerate(chunks):
    print(f"🔹 [Chunk {i+1}] ID: {c.chunk_id} | Vị trí: [{c.char_start}:{c.char_end}] | Độ dài: {len(c.chunk_text)} ký tự")
    print(f"\"{c.chunk_text}\"\n")                                                                                                                                                                                                            

NameError: name 'chunker' is not defined

In [93]:
def display_chunks(doc_id: int, text: str):
    chunks = chunker.chunk_document(doc_id=str(doc_id), text=text, lang="vi")
    # 4. Xem kết quả
    print(f"Tổng số chunks tạo ra: {len(chunks)}\n")
    for i, c in enumerate(chunks):
        print(f"🔹 [Chunk {i+1}] ID: {c.chunk_id} | Vị trí: [{c.char_start}:{c.char_end}] | Độ dài: {len(c.chunk_text)} ký tự")
        print(f"\"{c.chunk_text}\"\n")   

In [46]:
all_df = pd.concat([df, df1, df2, df3, df4, df5], ignore_index=True)

In [47]:
all_df

,doc_id,url,title,text,lang,domain,char_count,status
0,733125,https://www.vinmec.com/vie/bai-viet/-u-malt-da...,"U malt dạ dày Phần 1: Tổng quan, chẩn đoán và ...",Bài viết bởi Bác sĩ Mai Viễn Phương - Khoa Khá...,vi,www.vinmec.com,7110,success
1,2989609,https://www.a-hospital.com/w/%CE%91%E5%9C%B0%E...,地中海贫血,地中海贫血\n| A+医学百科 >> 地中海贫血 | \n地中海贫血又称海洋性贫血。是一组遗...,zh,www.a-hospital.com,7894,success
2,323036,https://suckhoecongdongonline.vn/0h-hom-nay-26...,0h hôm nay 26/7 công bố điểm thi THPT Quốc gia...,Tra cứu điểm thi THPT Quốc gia 2021\nSau 0h hô...,vi,suckhoecongdongonline.vn,3147,success
3,2474097,https://baidianfeng.familydoctor.com.cn/,鐨偆绉戦閬撀风櫧鐧滈,鐧界櫆椋庝笉鑳藉悆鍚淮c鐨勯鐗╋紵\n鐧界櫆椋庢槸涓�绉嶈壊绱犵己澶辩殑鐤剧梾锛屼富瑕佹槸鍥...,zh,baidianfeng.familydoctor.com.cn,3183,success
4,3984397,https://www.youlai.cn/yyk/article/136046.html,免疫性血小板减少性紫癜-有来医生,-视频-文章-语音\n免疫性血小板减少性紫癜\n2023-08-03 05:16:17阅读：...,zh,www.youlai.cn,825,success
...,...,...,...,...,...,...,...,...
447716,3539383,https://www.familydoctor.com.cn/baby/a/201512/...,鍎跨閬楀翱鐥囩殑椋熺枟鏂规硶鏈夊摢浜� 瑗垮尰濡備綍娌荤枟濂�,儿遗尿症的食疗方法有哪亄17 西医如何治疗奄17\n孩子大了老尿床，其实參昁尿症在作祟的，在...,zh,www.familydoctor.com.cn,1505,success
447717,2145335,https://ask.39.net/question/102529847.html,骶尾部藏毛窦手术后怎么大便,发病时间：不清楚\n补充说明：骶尾部藏毛窦手术后怎么大便\n我要咨询\n精选回答(1)\n藏...,zh,ask.39.net,472,success
447718,2998223,https://www.a-hospital.com/w/%E4%B8%B4%E6%B4%A...,临洮县人民医院,临洮县人民医院\n临洮县人民医院信息概要：\n- 医院地址：甘肃省定西地区临洮县新\n- 联...,zh,www.a-hospital.com,3721,success
447719,331646,https://suckhoecongdongonline.vn/bac-si-goc-vi...,Bác sĩ gốc Việt bị kéo lê trên máy bay Mỹ lần ...,Trong cuộc phỏng vấn hôm 9/4 trên chương trình...,vi,suckhoecongdongonline.vn,3846,success


In [67]:
vi_sentences = []
zh_sentences = []

for index, row in all_df.iterrows():
    text = row["text"]
    splits = text.split("\n")

    if row["lang"] == "vi":
        vi_sentences.append(
            {"text": row["text"], "url": row["url"], "title": row["title"]}
        )
    else:
        zh_sentences.append(
            {"text": row["text"], "url": row["url"], "title": row["title"]}
        )

In [68]:
len(vi_sentences)

227852

In [51]:
vi_sentences[-1]

{'text': 'Trong cuộc phỏng vấn hôm 9/4 trên chương trình Good Morning America, bác sĩ Mỹ gốc Việt David Dao cho hay dù sự việc xảy ra tròn 2 năm trước khiến mình và gia đình rất hoảng loạn, ông không hối tiếc vì đã giữ vững quan điểm và khiến hãng United Airlines phải xem xét lại các chính sách.\nNgày 9/4/2017, ông Dao vừa lên chuyến bay 3411 từ Chicago đi Louisville, bang Kentucky không lâu thì bất ngờ được thông báo rằng mình phải rời đi để nhường ghế cho các nhân viên của United Airlines, do chuyến bay bị đặt vé quá chỗ.\nTheo video do các hành khách khác quay lại, ông Dao bị các nhân viên an ninh dùng vũ lực lôi khỏi ghế ngồi dù đã thắt dây an toàn và bị kéo lê trên mặt đất giữa những tiếng la hét hoảng hốt. Một nhân chứng kể trong lúc bị lôi đi, mặt ông Dao va vào tay vịn ghế máy bay, khiến miệng ông chảy máu.\n10 phút sau đó, ông chạy ngược lại với gương mặt vẫn chảy máu, quần áo xộc xệch, mắt kính suýt rơi ra. Ông luôn miệng nói: "Họ sẽ giết tôi, tôi muốn về nhà". Các nhân viên 

In [52]:
all_vi_urls = [s.get("url") for s in vi_sentences]

In [69]:
all_zh_urls = [s.get("url") for s in zh_sentences]

In [53]:
len(vi_sentences)

227852

In [70]:
len(all_zh_urls)

219869

In [54]:
all_vi_urls

['https://www.vinmec.com/vie/bai-viet/-u-malt-da-day-phan-1-tong-quan-chan-doan-va-trieu-chung-vi',
 'https://suckhoecongdongonline.vn/0h-hom-nay-26-7-cong-bo-diem-thi-thpt-quoc-gia-2021',
 'https://suckhoedoisong.vn/1-2-so-nguoi-nhiem-omicron-co-the-khong-biet-ban-than-bi-nhiem-benh-169220818200054207.htm',
 'https://medlatec.vn/ba-bau-bi-sung-nuou-rang-trong-cung-nguyen-nhan-trieu-chung-va-cach-dieu-tri',
 'https://thanhnien.vn/-phu-huynh-anh-bo-quen-doi-mat-cua-con-nho-185179557.htm',
 'https://hellobacsi.com/an-uong-lanh-manh/beo-phi/beo-phi/',
 'https://giadinhonline.vn/1-nguoi-chet-18-nguoi-nhap-vien-sau-khi-an-co-o-thai-binh-bac-si-chi-mon-khoai-khau-mang-o-benh-d198531.html',
 'https://bachmai.gov.vn/bai-viet/-nguoi-dan-se-duoc-thu-huong-chat-luong-kham-chua-benh-quoc-te-ngay-tai-benh-vien-bach-mai--9867?id=da6d8b2d-c5bd-c612-e89c-e85f802063b4',
 'https://dantri.com.vn/doi-song/tin-vui-cho-nguoi-lon-tuoi-bi-sa-sut-tri-tue-20200210124423170.htm',
 'https://baohaiphong.vn/1-027-m

In [62]:
all_vi_urls[0].split("//")[1].split("/")[0]

'www.vinmec.com'

In [71]:
all_zh_urls[0].split("//")[1].split("/")[0]

'www.a-hospital.com'

In [78]:
zh_domains = []
for url in all_zh_urls:
    zh_domains.append(url.split("//")[1].split("/")[0])

In [79]:
set(zh_domains)

{'ask.39.net',
 'ask.familydoctor.com.cn',
 'baby.39.net',
 'bachmai.gov.vn',
 'baidianfeng.familydoctor.com.cn',
 'baodanang.vn',
 'byby.39.net',
 'cancer.39.net',
 'care.39.net',
 'ek.39.net',
 'fitness.39.net',
 'fk.39.net',
 'fk.99.com.cn',
 'food.39.net',
 'gan.39.net',
 'gc.39.net',
 'gk.39.net',
 'health.people.com.cn',
 'heart.39.net',
 'hellobacsi.com',
 'jb39.com',
 'jbk.39.net',
 'man.39.net',
 'nk.39.net',
 'pf.39.net',
 'qihuangzhishu.com',
 'qy.familydoctor.com.cn',
 'shen.39.net',
 'sj.39.net',
 'suckhoedoisong.vn',
 'suckhoeviet.org.vn',
 'test.pmphai.com',
 'thaythuocvietnam.vn',
 'tnb.39.net',
 'v.familydoctor.com.cn',
 'wei.39.net',
 'woman.39.net',
 'www.120ask.com',
 'www.a-hospital.com',
 'www.baidu.com',
 'www.cnkang.com',
 'www.familydoctor.cn',
 'www.familydoctor.com.cn',
 'www.msdmanuals.cn',
 'www.vinmec.com',
 'www.wujue.com',
 'www.youlai.cn',
 'www.zydcd.com',
 'yanglao.familydoctor.com.cn',
 'ypk.familydoctor.com.cn',
 'zhongyibaodian.net'}

In [75]:
vi_domains = []
for url in all_vi_urls:
    vi_domains.append(url.split("//")[1].split("/")[0])

In [76]:
set(vi_domains)

{'bachmai.gov.vn',
 'baoangiang.com.vn',
 'baocantho.com.vn',
 'baochinhphu.vn',
 'baodanang.vn',
 'baogialai.com.vn',
 'baohaiphong.vn',
 'baolangson.vn',
 'baonghean.vn',
 'baophutho.vn',
 'baoquangninh.vn',
 'baoquangtri.vn',
 'baotayninh.vn',
 'baothanhhoa.vn',
 'baovinhlong.com.vn',
 'benhviennhitrunguong.gov.vn',
 'benhvienvietduc.org',
 'dantri.com.vn',
 'giadinhonline.vn',
 'hanoimoi.vn',
 'hellobacsi.com',
 'khoahocphothong.vn',
 'medlatec.vn',
 'mega.vietnamplus.vn',
 'nhandan.vn',
 'phunusuckhoe.giadinhonline.vn',
 'phuyen.baodaklak.vn',
 'special.vietnamplus.vn',
 'suckhoecongdongonline.vn',
 'suckhoedoisong.vn',
 'suckhoeviet.org.vn',
 'thanhnien.vn',
 'thaythuocvietnam.vn',
 'tiemchunglongchau.com.vn',
 'tienphong.vn',
 'tuoitre.vn',
 'vietnamnet.vn',
 'vnexpress.net',
 'vov2.vov.vn',
 'www.pharmacity.vn',
 'www.qdnd.vn',
 'www.sggp.org.vn',
 'www.vietnamplus.vn',
 'www.vinmec.com',
 'youmed.vn'}

In [ ]:
vi_sentences[10]

{'text': 'Tiêm vắc xin Menactra là một giải pháp giúp bảo vệ cơ thể khỏi các bệnh do vi khuẩn não mô cầu Neisseria meningitidis gây ra, bao gồm viêm màng não, nhiễm trùng huyết và viêm phổi. Vắc xin viêm màng não này đặc biệt hiệu quả trong việc phòng ngừa các chủng vi khuẩn thuộc nhóm huyết thanh A, C, Y, W-135, những chủng có thể gây bệnh nghiêm trọng nếu không được điều trị kịp thời.\n1 mũi tiêm vắc xin Menactra giá bao nhiêu?\nVắc xin Menactra tại Trung tâm Tiêm chủng Long Châu có giá 1.360.000 VNĐ, nhưng có thể thay đổi tùy thời điểm. Tuy nhiên, mức giá này có thể thay đổi tùy vào thời điểm, tình trạng cung ứng vắc xin và các chương trình ưu đãi tại các cơ sở tiêm chủng. Do vậy, nếu bạn đang lên kế hoạch tiêm vắc xin này, hãy liên hệ hotline 18006928 để được tư vấn cụ thể và chính xác nhất nhé!\nKhi bạn quyết định tiêm vắc xin Menactra để phòng ngừa các bệnh do vi khuẩn não mô cầu Neisseria meningitidis gây ra, một trong những yếu tố mà bạn chắc chắn sẽ quan tâm là giá của loại vắ

In [91]:
all_vi_title = [t.get("title") for t in vi_sentences if t.get("title") != "Trang chủ"]

In [88]:
homepage_docs = [t for t in vi_sentences if t.get("title") == "Trang chủ"]

In [89]:
homepage_docs

[{'text': 'Thầy Thuốc Việt Nam\nTrang cộng đồng\nTrang bác sĩ\nHotline\n0384371974\nY học cơ sở\nY học lâm sàng\nPhác đồ\nVideo\nThuốc và biệt dược\nTạp chí\nThông tin hữu ích\nĐK thành viên\nTrang chủ\nGiới thiệu\nBệnh thường gặp\nVideo sức khỏe ngắn\nChia sẻ trực tuyến\nKhám bệnh Online\nLiên hệ hợp tác\nMenu\nĐóng\nTrang chủ\nGiới thiệu\nBệnh thường gặp\nVideo sức khỏe ngắn\nChia sẻ trực tuyến\nKhám bệnh Online\nLiên hệ hợp tác\nTrang bác sĩ\nBài viết nổi bật\nĐọc nhiều nhất\nBài viết mới nhất\nDA LIỄU\n10 cách trị mụn trứng cá ở tuổi dậy thì an toàn và hiệu quả\n02/04/2023\nDINH DƯỠNG\n10 loại thực phẩm tốt cho tim mạch\n14/02/2023\nTIÊU HÓA\n10+ cách trị táo bón cho người lớn đơn giản, mau khỏi\n14/03/2023\nDA LIỄU\n12 cách chữa viêm da cơ địa tại nhà hiệu quả, an toàn, tiết kiệm\n19/03/2023\nTHẦN KINH\n12 mẹo cải thiện trí nhớ cho người hay quên!\n15/03/2023\nDA LIỄU\n13 Cách trị mụn tại nhà từ thiên nhiên hiệu quả chỉ sau 1 tuần thực hiện\n27/03/2023\nDINH DƯỠNG\n5 Bí quyết để c

In [90]:
len(homepage_docs)

2230

In [92]:
all_vi_title

['U malt dạ dày Phần 1: Tổng quan, chẩn đoán và triệu chứng',
 '0h hôm nay 26/7 công bố điểm thi THPT Quốc gia 2021 | Sức Khỏe Cộng Đồng',
 '1/2 số người nhiễm Omicron có thể không biết bản thân bị nhiễm bệnh',
 'Bà bầu bị sưng nướu răng trong cùng: Nguyên nhân, triệu chứng và cách điều trị',
 '¾ phụ huynh Anh “bỏ quên” đôi mắt của con nhỏ',
 'Béo phì • Hello Bacsi',
 '1 người chết, 18 người nhập viện sau khi ăn cỗ ở Thái Bình: Bác sĩ chỉ món khoái khẩu mang ổ bệnh',
 '“Người dân sẽ được thụ hưởng chất lượng khám chữa bệnh Quốc tế ngay tại Bệnh viện Bạch Mai”',
 'Tin vui cho người lớn tuổi bị sa sút trí tuệ',
 '1.027 mẫu liên quan đến Bệnh viện Phụ sản có kết quả xét nghiệm âm tính với vi-rút SARS-CoV-2',
 '1 mũi tiêm vắc xin Menactra giá bao nhiêu tiền?',
 '1/6 bệnh nhân mắc ung thư đều do viêm: 5 vị trí trên cơ thể bị viêm cảnh báo ung thư đang đến gần',
 'Ðảm bảo học sinh, sinh viên được chăm sóc sức khỏe toàn diện',
 "1.000 người tham gia ngày 'Toàn dân hiến máu tình nguyện'",
 '‘B

In [94]:
def display_chunks(doc_id: int, text: str):
    chunks = chunker.chunk_document(doc_id=str(doc_id), text=text, lang="vi")
    # 4. Xem kết quả
    print(f"Tổng số chunks tạo ra: {len(chunks)}\n")
    for i, c in enumerate(chunks):
        print(f"🔹 [Chunk {i+1}] ID: {c.chunk_id} | Vị trí: [{c.char_start}:{c.char_end}] | Độ dài: {len(c.chunk_text)} ký tự")
        print(f"\"{c.chunk_text}\"\n")   

In [108]:
display_chunks("1", vi_sentences[1000]["text"])

Tổng số chunks tạo ra: 4

🔹 [Chunk 1] ID: 1__c0 | Vị trí: [0:1200] | Độ dài: 1199 ký tự
"5 điều cần biết để ăn hải sản không bị ngộ độc
Khi đi biển nghỉ mát và thưởng thức hải sản, ăn thế nào cũng là vấn đề quan trọng, Trung tâm chống độc Bệnh viện Bạch Mai có hướng dẫn như sau:
1. Chỉ ăn loại hải sản được biết chắc chắn không có chất độc:
- Đây chính là các loại hải sản chúng ta vẫn thường ăn. Điều này không phải vì yếu tố “tâm lý đám đông”, các hải sản đó rõ ràng an toàn.
- Thận trọng khi ăn các loại hải sản lạ, ít khi được ăn. Đây thực sự là sở thích khám phá của nhiều người. Tuy nhiên, cần cân nhắc trước khi ăn thử. Lý do các loại hải sản này ít được ăn cũng có thể từng được biết là có thể gây ngộ độc, hoặckhông ai biết có gây ngộ độc hay không. Vì vậy đây cũng là nhóm hải sản có thể gây ngộ độc và nên tránh.
2. Biết và tránh các loại hải sản được biết có thể chứa chất độc:
Có loại hải sản luôn luôn có chất độc, có loại thường có hoặc thỉnh thoảng mới có. Tuy nhiên, bằng mắt thường

In [111]:
corpus_urls = []
for index, row in corpus.iterrows():
    corpus_urls = row['url']


In [112]:
set(corpus_urls)

{'.',
 '/',
 ':',
 'a',
 'b',
 'c',
 'e',
 'h',
 'i',
 'j',
 'l',
 'm',
 'n',
 'o',
 'p',
 'r',
 's',
 't',
 'u',
 'y',
 'z'}